# Week 51, Day 5: Controls: is the winning model right for the right reasons?

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w051/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.signal import lfilter
d = np.load(dataset("ibl_biased.npz"))
mouse, session = d["mouse"].astype(int), d["session"].astype(int)
contrast, choice, reward = d["contrast"].astype(float), d["choice"].astype(float), d["reward"].astype(int)
p_left, rt = d["p_left"].astype(float).round(2), d["rt"].astype(float)
ok = (choice >= 0) & d["included"] & np.isfinite(rt)
sess = 10 * mouse + session
f_c = np.tanh(5 * contrast) / np.tanh(5)
c_pm = np.where(choice >= 0, 2 * choice - 1, 0.0)
side = np.where(contrast != 0, np.sign(contrast), np.where(reward == 1, c_pm, -c_pm))

def kernel(x, alpha, sess_ids):
    out = np.zeros(len(x))
    for s in np.unique(sess_ids):
        k = sess_ids == s
        out[k] = lfilter([alpha], [1, -(1 - alpha)], np.r_[0, x[k][:-1]])
    return out

def lag1(x, sess_ids):
    out = np.r_[0, x[:-1]]; out[np.r_[True, sess_ids[1:] != sess_ids[:-1]]] = 0
    return out

def bayes_prior(side, sess_ids, h=0.02):
    out = np.zeros(len(side))
    for s in np.unique(sess_ids):
        b = np.array([0.5, 0.5])
        for t in np.flatnonzero(sess_ids == s):
            pr = 0.8 * b[0] + 0.2 * b[1]; out[t] = np.log(pr / (1 - pr))
            b = b * (np.array([0.8, 0.2]) if side[t] > 0 else np.array([0.2, 0.8])); b = b / b.sum()
            b = np.array([(1 - h) * b[0] + h * b[1], (1 - h) * b[1] + h * b[0]])
    return out

prior_bayes = bayes_prior(side, sess)
models = ["bayes", "stimulus", "action"]
x0 = {"bayes": [3, 0, 1], "stimulus": [3, 0, 1, 0.2], "action": [3, 0, 1, 0.2]}
bounds = {"bayes": [(-20, 20)] * 3, "stimulus": [(-20, 20)] * 3 + [(0.01, 1)], "action": [(-20, 20)] * 3 + [(0.01, 1)]}

def logit(model, p, k, actions):
    """Logit of P(right) on the trials selected by k, given the +-1 actions that drive an action kernel."""
    z = p[0] * f_c[k] + p[1]
    if model == "bayes":
        return z + p[2] * prior_bayes[k]
    x = side[k] if model == "stimulus" else actions
    return z + p[2] * kernel(x, p[3], sess[k])

def fit(model, k, y_all, extra=None):
    """Fit a model to the trials selected by k, with choices y_all (0/1, all trials of k). Returns (params, max log-lik)."""
    use = ok[k]; y = y_all[use]; a = np.where(y_all >= 0, 2 * y_all - 1, 0.0)
    n_extra = 0 if extra is None else 1
    def f(p):
        z = logit(model, p, k, a)
        if extra is not None:
            z = z + p[-1] * extra
        z = z[use]
        return np.sum(np.logaddexp(0, z) - y * z)
    res = minimize(f, x0[model] + [0.0] * n_extra, method="L-BFGS-B", bounds=bounds[model] + [(-20, 20)] * n_extra)
    return res.x, -res.fun

params, loglik = {}, np.zeros((18, 3))
for m in range(18):
    k = mouse == m
    for j, mod in enumerate(models):
        params[m, mod], loglik[m, j] = fit(mod, k, choice[k])
n_m = np.array([np.sum(ok & (mouse == m)) for m in range(18)])
print("refitted three models to 18 mice; action kernel best for", np.sum(loglik.argmax(1) == 2), "mice by likelihood")

### Exercise: Giving every model the previous choice

Compute prev, each trial's previous choice (+1, -1 or 0) within its session. Then refit every model with prev as an extra regressor and store -BIC/2 (counting the extra parameter) in lme_p.

In [ ]:
prev = ...
lme_p = np.zeros((18, 3))
for m in range(18):
    k = mouse == m
    for j, mod in enumerate(models):
        p, L = fit(mod, k, choice[k], extra=prev[k])
        lme_p[m, j] = ...
print("with one-trial perseveration in every model, best model counts:", {mod: int(np.sum(lme_p.argmax(1) == j)) for j, mod in enumerate(models)})
print("summed evidence relative to the best:", np.round(lme_p.sum(0) - lme_p.sum(0).max(), 1))

In [ ]:
import numpy as np
def test_shape():
    "lme_p has one row per mouse and one column per model, using BIC with the extra parameter"
    assert lme_p.shape == (18, 3) and np.all(np.isfinite(lme_p))
def test_bic():
    "each entry is -BIC/2 with one more parameter than the base model"
    m = 2; k = mouse == m; _, L = fit("bayes", k, choice[k], extra=prev[k])
    assert abs(lme_p[m, 0] - (L - 4 * np.log(n_m[m]) / 2)) < 1e-3
def test_winner():
    "the action kernel still wins for at least 15 mice"
    assert np.sum(lme_p.argmax(1) == 2) >= 15

test_shape()
test_bic()
test_winner()
print("All checks passed")

### Exercise: Who repeats after errors?

Complete simulate. On each trial compute the logit from the stimulus, the bias and the model's prior; draw the choice; and, for the action kernel, update the running kernel with the simulated choice (resetting it at the start of each session). The GLM fit is written for you.

In [ ]:
def simulate(model, p, m, rng):
    k = np.flatnonzero(mouse == m)
    fc, sd, ss = f_c[k], side[k], sess[k]
    fixed = prior_bayes[k] if model == "bayes" else kernel(sd, p[3], ss) if model == "stimulus" else None
    y = np.zeros(len(k)); kern = 0.0
    for i in range(len(k)):
        if i == 0 or ss[i] != ss[i - 1]:
            kern = 0.0
        prior = kern if model == "action" else fixed[i]
        z = ...
        y[i] = rng.random() < 1 / (1 + np.exp(-z))
        if model == "action":
            kern = ...
    return y

def after_weights(y, m):
    """Day 2's GLM fitted to choices y of mouse m: returns (after-reward, after-error) weights."""
    k = mouse == m; a = 2 * y - 1; rew = (a == side[k]).astype(float); use = ok[k]
    X = np.c_[f_c[k], lag1(a * rew, sess[k]), lag1(a * (1 - rew), sess[k]), np.ones(k.sum())][use]; yy = y[use]
    w = np.zeros(4)
    for _ in range(25):
        q = 1 / (1 + np.exp(-X @ w))
        w -= np.linalg.solve(-(X * (q * (1 - q))[:, None]).T @ X - 1e-3 * np.eye(4), X.T @ (yy - q) - 1e-3 * w)
    return w[1], w[2]

rng = np.random.default_rng(1)
win, lose = {}, {}
for mod in ["data"] + models:
    ws = np.array([after_weights(choice[mouse == m].clip(0) if mod == "data" else simulate(mod, params[m, mod], m, rng), m) for m in range(18)])
    win[mod], lose[mod] = ws[:, 0], ws[:, 1]
    print(f"{mod:9s} after reward {win[mod].mean():+.2f}, after error {lose[mod].mean():+.2f} (positive for {np.sum(lose[mod] > 0)} of 18)")

In [ ]:
import numpy as np
def test_sim():
    "simulate returns 0/1 choices for every trial of the mouse"
    a = simulate("action", params[0, "action"], 0, np.random.default_rng(0))
    assert a.shape == (np.sum(mouse == 0),) and set(np.unique(a)) <= {0.0, 1.0}
def test_signature():
    "simulated action-kernel mice repeat after errors, simulated Bayesian and stimulus-kernel mice do not"
    assert np.sum(lose["action"] > 0) >= 15 and np.sum(lose["bayes"] > 0) <= 3 and np.sum(lose["stimulus"] > 0) <= 3

test_sim()
test_signature()
print("All checks passed")

### Exercise: Model recovery with realistic parameters

For each generating model and each mouse in subset, simulate choices, fit all three models to them, and add one to conf at (generating model, model with the highest -BIC/2). Normalise each row.

In [ ]:
subset = [0, 3, 6, 9, 12, 15]
conf = np.zeros((3, 3))
rng = np.random.default_rng(2)
for g, gen in enumerate(models):
    for m in subset:
        k = mouse == m
        y = simulate(gen, params[m, gen], m, rng)
        scores = [fit(mod, k, y)[1] - len(x0[mod]) * np.log(n_m[m]) / 2 for mod in models]
        ...
conf = conf / conf.sum(1, keepdims=True)
print("rows: generating model; columns: model selected")
for g, gen in enumerate(models):
    print(f"{gen:9s}", np.round(conf[g], 2))

In [ ]:
import numpy as np
def test_conf():
    "conf is a 3 by 3 matrix of fractions whose rows sum to 1"
    assert conf.shape == (3, 3) and np.allclose(conf.sum(1), 1)
def test_diag():
    "each model is recovered for at least two thirds of its simulated mice"
    assert np.all(np.diag(conf) >= 2 / 3)

test_conf()
test_diag()
print("All checks passed")

### Exercise: A fast and a slow kernel

For each mouse, compute the slow kernel of its choices with rate 0.02, refit the action model with it as an extra regressor, and store the old fast weight, the new fast weight, the new fast rate, the slow weight and the gain in log-likelihood.

In [ ]:
drift_res = np.zeros((18, 5))
for m in range(18):
    k = mouse == m
    slow = ...
    p2, L2 = fit("action", k, choice[k], extra=slow)
    drift_res[m] = ...
print(f"fast weight keeps a median {np.median(drift_res[:, 1] / drift_res[:, 0]):.0%} of its value; fast rate now {np.median(drift_res[:, 2]):.2f}")
print(f"slow weight: median {np.median(drift_res[:, 3]):.2f}; gain in log-likelihood: median {np.median(drift_res[:, 4]):.1f} nats")
print(f"mice for which the slow kernel is worth its parameter by BIC: {np.sum(drift_res[:, 4] > np.log(n_m) / 2)} of 18")

In [ ]:
import numpy as np
def test_cols():
    "drift_res has, per mouse, the fast weight before and after, the new fast rate, the slow weight and the gain in log-likelihood"
    assert drift_res.shape == (18, 5) and np.all(drift_res[:, 4] > -1e-3)
def test_fast():
    "the fast kernel keeps most of its weight in the median mouse"
    assert np.median(drift_res[:, 1] / drift_res[:, 0]) > 0.5

test_cols()
test_fast()
print("All checks passed")